# E v8：75 条已定案 NQ Train 开发样本

选择 Kaggle T4 GPU，打开联网，添加配套 tar.gz 为输入，依次运行全部单元。
也支持 Colab T4 上传。四组：v7 对照、v8 完整流程、去原答案核验、去候选核验。
每组同 75 条，共 300 条输出，最多 600 次 Qwen 生成。C1 直接复用原始结果，零新增调用。
11 条 E 目标与 64 条正确对照只用于开发；25 条待定不在包内。同模型核验不是独立裁判。
结束后下载导出的 ZIP；语义收益留待人工审核，不由 EM 或核验通过自动认定。


Transport r1：兼容 Kaggle 自动解压输入；原 906f096f 冻结包、75 条样本和四组算法完全相同。


In [ ]:
import json, os, sys, subprocess, hashlib, tarfile, shutil
from pathlib import Path
import torch
assert sys.version_info >= (3, 11), 'Python 3.11+ required'
assert torch.cuda.is_available(), 'Select a GPU runtime'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
EXPECTED_SHA256 = '906f096f78e36a4f272cdd863ce2e7383d62acb638c1e480a06c5d3f75daa7a3'
BUNDLE_NAME = 'answer_repair_v8_nq75_gpu'
OUTPUT = WORK / (BUNDLE_NAME.removesuffix('_gpu') + '_outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)


In [ ]:
"""Stdlib bootstrap for intact archives and Kaggle's automatically extracted inputs.

This source is embedded in the transport notebook before the frozen code exists
on disk. The checksums document itself is pinned outside the uploaded directory.
"""
from __future__ import annotations

import hashlib
import json
import shutil
import tarfile
from pathlib import Path, PurePosixPath


def _digest(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _verified_files(root: Path, expected_checksums_sha256: str) -> dict[str, str]:
    checksum_path = root / "bundle_checksums.json"
    if _digest(checksum_path) != expected_checksums_sha256:
        raise ValueError("Frozen checksums document hash mismatch")
    files = json.loads(checksum_path.read_bytes())
    if not isinstance(files, dict) or not files:
        raise ValueError("Invalid checksums document")
    for name, digest in files.items():
        path = PurePosixPath(name)
        if (not name or path.is_absolute() or ".." in path.parts or "\\" in name
                or not isinstance(digest, str) or len(digest) != 64):
            raise ValueError("Invalid frozen file entry")
        target = root / name
        if not target.resolve().is_relative_to(root.resolve()) or not target.is_file() or target.is_symlink():
            raise ValueError(f"Frozen file missing or unsafe: {name}")
        if _digest(target) != digest:
            raise ValueError(f"Frozen file hash mismatch: {name}")
    return files


def load_frozen_bundle(*, work: Path, input_root: Path, bundle_name: str,
                       archive_sha256: str, checksums_sha256: str) -> tuple[Path, Path | None, dict]:
    """Copy only verified payloads to a writable directory, without changing them."""
    if not bundle_name.replace("_", "").isalnum():
        raise ValueError("Unsafe bundle name")
    archive_name = bundle_name + ".tar.gz"
    local_archive = work / archive_name
    archives = ([local_archive] if local_archive.is_file() else
                sorted(input_root.rglob(archive_name)) if input_root.exists() else [])
    if len(archives) > 1:
        raise ValueError("Multiple matching archives; attach one frozen bundle")
    staging = work / ("e_v8_" + archive_sha256[:12])
    destination = staging / bundle_name
    archive = archives[0] if archives else None
    if archive is not None:
        if _digest(archive) != archive_sha256:
            raise ValueError("Archive hash mismatch")
        if not destination.exists():
            staging.mkdir(parents=True, exist_ok=True)
            with tarfile.open(archive) as handle:
                handle.extractall(staging, filter="data")
        files = _verified_files(destination, checksums_sha256)
        mode = "intact_archive"
        source = str(archive)
    else:
        roots = ([p.parent for p in input_root.rglob("bundle_checksums.json")
                  if p.parent.name == bundle_name] if input_root.exists() else [])
        if len(roots) != 1:
            raise ValueError(f"Expected one archive or auto-extracted bundle, found {len(roots)} extracted roots")
        source_root = roots[0]
        files = _verified_files(source_root, checksums_sha256)
        destination.mkdir(parents=True, exist_ok=True)
        # Ignore unverified extra files rather than adding them to Python's import path.
        for name in [*files, "bundle_checksums.json"]:
            source_file, target = source_root / name, destination / name
            if target.exists():
                if _digest(target) != _digest(source_file):
                    raise ValueError(f"Existing writable bundle differs: {name}")
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(source_file, target)
        _verified_files(destination, checksums_sha256)
        mode, source = "kaggle_auto_extracted", str(source_root)
    report = {"transport": mode, "source": source, "bundle": str(destination),
              "expected_archive_sha256": archive_sha256,
              "archive_hash_verified": archive is not None,
              "checksums_sha256": checksums_sha256, "payload_hash_checked_files": len(files),
              "algorithm_or_input_modified": False}
    return destination, archive, report

# Transport r1; original frozen payload and protocol are unchanged.
EXPECTED_CHECKSUMS_SHA256 = '5499fcca237c5d8f50187c27bf24f5f6189176dab8d86184b507116420f3997e'
if not Path('/kaggle/input').exists() and not (WORK / (BUNDLE_NAME + '.tar.gz')).exists():
    from google.colab import files
    uploaded = files.upload()
    upload_name = BUNDLE_NAME + '.tar.gz'
    assert upload_name in uploaded, 'Upload the matching tar.gz'
    (WORK / upload_name).write_bytes(uploaded[upload_name])
try:
    BUNDLE, archive, bootstrap_report = load_frozen_bundle(
        work=WORK, input_root=Path('/kaggle/input'), bundle_name=BUNDLE_NAME,
        archive_sha256=EXPECTED_SHA256, checksums_sha256=EXPECTED_CHECKSUMS_SHA256)
except Exception as exc:
    (OUTPUT / 'bootstrap_error.json').write_text(json.dumps({'error': str(exc)}) + '\n')
    raise
(OUTPUT / 'bootstrap.json').write_text(json.dumps(bootstrap_report, indent=2) + '\n')
manifest = json.loads((BUNDLE / 'manifest.json').read_text())
assert manifest['samples'] == 75
for source in ('manifest.json', 'bundle_checksums.json'):
    shutil.copy2(BUNDLE / source, OUTPUT / source)
if archive is not None:
    shutil.copy2(archive, OUTPUT / archive.name)
print('Bundle verified:', bootstrap_report)


In [ ]:
def run_logged(argv, name):
    with (OUTPUT / name).open('a', encoding='utf-8') as log:
        log.write('COMMAND ' + json.dumps(argv) + '\n'); log.flush()
        process = subprocess.Popen(argv, cwd=BUNDLE, env={**os.environ, 'PYTHONUNBUFFERED': '1'},
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if status: raise subprocess.CalledProcessError(status, argv)

run_logged([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-answer-repair-v8.txt'], 'install.log')
run_logged([sys.executable, '-m', 'pip', 'freeze'], 'packages.log')
run_logged([sys.executable, 'scripts/check_answer_repair_environment.py', '--require-cuda',
            '--out', str(OUTPUT / 'environment.json')], 'environment.log')
run_logged([sys.executable, '-m', 'unittest', *manifest['portable_tests']], 'tests.log')
RUNS = []
for arm in manifest['protocol']:
    strategy = arm['strategy']
    argv = [sys.executable, '-u', 'scripts/run_answer_repair.py', '--dataset', 'nq',
            '--c1', str(BUNDLE / 'inputs/nq75_c1.jsonl'), '--all', '--strategy', strategy,
            '--max-new-tokens', str(arm['max_new_tokens']), '--out', str(OUTPUT / (strategy + '.jsonl'))]
    RUNS.append((strategy, argv))
    run_logged(argv + ['--check-only'], strategy + '_preflight.log')


依次完成四组，不根据中途结果改提示或挑选样本。
中断后可在相同环境续跑；已完成的预测会校验并跳过。更换环境需保留旧输出，另开新实验目录。


In [ ]:
try:
    for strategy, argv in RUNS:
        run_logged(argv, strategy + '_run.log')
finally:
    # Preserve partial progress even if a backend fails; errors still stop the experiment.
    partial = shutil.make_archive(str(WORK / (OUTPUT.name + '_partial')), 'zip', OUTPUT)
    print('Recovery archive:', partial)


In [ ]:
run_logged([sys.executable, 'scripts/summarize_answer_repair_v8.py',
            '--bundle', str(BUNDLE), '--results', str(OUTPUT)], 'comparison.log')
report = json.loads((OUTPUT / 'comparison.json').read_text())
assert report['all_arms_complete'], 'Incomplete experiment; preserve partial output'
export = shutil.make_archive(str(WORK / (OUTPUT.name + '_export')), 'zip', OUTPUT)
print('Result archive:', export)
print('SHA256:', hashlib.sha256(Path(export).read_bytes()).hexdigest())
print('Changed outputs pending review:', report['pending_changed_outputs'])
if not Path('/kaggle/working').exists():
    from google.colab import files
    files.download(export)
